In [3]:
# Install libraries
# datasets < 4.0.0 is required: newer versions dropped support for
# "loading scripts", which is what tner/ontonotes5 is built on

!pip install -q --upgrade pip setuptools wheel
!pip install -q numpy
!pip install -q seqeval
!pip install -q "datasets<4.0.0"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
^C
  Preparing metadata (pyproject.toml) ... canceled
ERROR: Operation cancelled by user


In [3]:
# Load the dataset
# tner/ontonotes5: a "flat" version of OntoNotes 5.0 (just tokens/tags,
# no nested document structure). 18 entity types, BIO tagging scheme

from datasets import load_dataset

ds = load_dataset("tner/ontonotes5")
print(ds)

DatasetDict({
    train: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 59924
    })
    validation: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 8528
    })
    test: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 8262
    })
})


In [5]:
# Label list
# Written out manually: the parquet auto-conversion of this dataset
# strips label names (.features["tags"].feature.names raises AttributeError).
# Verified against the official label2id.json from the tner project — exact match.

label_list = [
    "O", "B-CARDINAL", "B-DATE", "I-DATE", "B-PERSON", "I-PERSON", "B-NORP", "B-GPE", "I-GPE",
    "B-LAW", "I-LAW", "B-ORG", "I-ORG", "B-PERCENT", "I-PERCENT", "B-ORDINAL", "B-MONEY", "I-MONEY",
    "B-WORK_OF_ART", "I-WORK_OF_ART", "B-FAC", "B-TIME", "I-CARDINAL", "B-LOC", "B-QUANTITY",
    "I-QUANTITY", "I-NORP", "I-LOC", "B-PRODUCT", "I-TIME", "B-EVENT", "I-EVENT", "I-FAC",
    "B-LANGUAGE", "I-PRODUCT", "I-ORDINAL", "I-LANGUAGE"
]
print(len(label_list))

37


In [6]:
example = ds["train"][0]
for tok, tag_id in zip(example["tokens"], example["tags"]):
    print(tok, "->", label_list[tag_id])

People -> O
start -> O
their -> O
own -> O
businesses -> O
for -> O
many -> O
reasons -> O
. -> O


In [7]:
for i in range(500):
    ex = ds["train"][i]
    tags = [label_list[t] for t in ex["tags"]]
    if any(t in ("B-DATE", "B-GPE", "B-LOC") for t in tags):
        print("Приклад №", i)
        for tok, t in zip(ex["tokens"], tags):
            print(tok, "->", t)
        break

Приклад № 6
Last -> B-DATE
week -> I-DATE
, -> O
Sen. -> O
Malcolm -> B-PERSON
Wallop -> I-PERSON
-LRB- -> O
R. -> B-NORP
, -> O
Wyo -> B-GPE
. -> I-GPE
-RRB- -> O
held -> O
hearings -> O
on -> O
a -> O
bill -> O
to -> O
strengthen -> O
an -> O
existing -> O
law -> O
designed -> O
to -> O
reduce -> O
regulatory -> O
hassles -> O
for -> O
small -> O
businesses -> O
. -> O


In [13]:
print(f"train: {len(ds["train"])}, val: {len(ds["validation"])}, test: {len(ds["test"])}")

train: 59924, val: 8528, test: 8262


In [16]:
# Custom train/val/test split (80/10/10)

# The dataset ships with official splits, but we chose to split it
# ourselves: pool everything together and re-cut with a fixed seed

# IMPORTANT: this dataset has no document id, so a group split
# (keeping sentences from the same document out of both train and
# test) isn't possible here — a plain random split is the correct
# choice given this input data

from datasets import concatenate_datasets

all_data = concatenate_datasets([ds["train"], ds["validation"], ds["test"]])
print(f"Count of all sentences: {len(all_data)}")

split1 = all_data.train_test_split(test_size=0.2, seed=42)
split2 = split1["test"].train_test_split(test_size=0.5, seed=42)

my_train = split1["train"]
my_val = split2["train"]
my_test = split2["test"]

print(f"train: {len(my_train)}, val: {len(my_val)}, test: {len(my_test)}")

Всього прикладів: 76714
train: 61371, val: 7671, test: 7672


In [20]:
# Code for original DS

#new_label_list = ["O", "B-LOC", "I-LOC", "B-DATE", "I-DATE"]
#new_label2id = {l: i for i, l in enumerate(new_label_list)}

#def remap(old_name):
 #   if old_name in ("B-GPE", "B-LOC"):
  #      return "B-LOC"
   # if old_name in ("I-GPE", "I-LOC"):
    #    return "I-LOC"
    #if old_name == "B-DATE":
     #   return "B-DATE"
    #if old_name == "I-DATE":
     #   return "I-DATE"
    #return "O"

#old_id_to_new_id = {old_id: new_label2id[remap(name)] for old_id, name in enumerate(label_list)}

#def remap_tags(example):
 #   example["tags"] = [old_id_to_new_id[t] for t in example["tags"]]
  #  return example

#ds_remapped = ds.map(remap_tags)
#print(ds_remapped["train"][6]) 


In [24]:
# Collapse 37 labels down to 5 (O, B-LOC, I-LOC, B-DATE, I-DATE)

new_label_list = ["O", "B-LOC", "I-LOC", "B-DATE", "I-DATE"]
new_label2id = {l: i for i, l in enumerate(new_label_list)}

def remap(old_name):
    if old_name in ("B-GPE", "B-LOC"):
        return "B-LOC"
    if old_name in ("I-GPE", "I-LOC"):
        return "I-LOC"
    if old_name == "B-DATE":
        return "B-DATE"
    if old_name == "I-DATE":
        return "I-DATE"
    return "O"

# Lookup table: old numeric id -> new numeric id
old_id_to_new_id = {old_id: new_label2id[remap(name)] for old_id, name in enumerate(label_list)}

def remap_tags(example):
    example["tags"] = [old_id_to_new_id[t] for t in example["tags"]]
    return example

my_train = my_train.map(remap_tags)
my_val = my_val.map(remap_tags)
my_test = my_test.map(remap_tags)



Map:   0%|          | 0/61371 [00:00<?, ? examples/s]

Map:   0%|          | 0/7671 [00:00<?, ? examples/s]

Map:   0%|          | 0/7672 [00:00<?, ? examples/s]

In [29]:
print(my_train[0])

{'tokens': ['because', 'I', "'ve", 'seen', 'a', 'map', '.'], 'tags': [0, 0, 0, 0, 0, 0, 0]}


In [30]:
# Tokenization + label alignment to subwords
from transformers import AutoTokenizer

# bert-base-cased: "cased" specifically (keeps letter casing) — an
# important signal for NER, since capitalization often hints that a
# word is a named entity

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

def tokenize_and_align_labels(example):
    tokenized = tokenizer(example["tokens"], is_split_into_words=True, truncation=True)
    word_ids = tokenized.word_ids()
    labels, prev = [], None
    for wid in word_ids:
        if wid is None:
            labels.append(-100)
        elif wid != prev:
            labels.append(example["tags"][wid])
        else:
            labels.append(-100)
        prev = wid
    tokenized["labels"] = labels
    return tokenized

my_train = my_train.map(tokenize_and_align_labels)
my_val = my_val.map(tokenize_and_align_labels)
my_test = my_test.map(tokenize_and_align_labels)

print(my_train[0])

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/61371 [00:00<?, ? examples/s]

Map:   0%|          | 0/7671 [00:00<?, ? examples/s]

Map:   0%|          | 0/7672 [00:00<?, ? examples/s]

{'tokens': ['because', 'I', "'ve", 'seen', 'a', 'map', '.'], 'tags': [0, 0, 0, 0, 0, 0, 0], 'input_ids': [101, 1272, 146, 112, 1396, 1562, 170, 4520, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [-100, 0, 0, 0, -100, 0, 0, 0, 0, -100]}


In [1]:
# Sanity check: scan the first 200 examples and print the first one
# that contains an actual entity label (not just O), so we can
# visually confirm the label alignment looks correct on a real case

for i in range(200):
    ex = my_train[i]
    if any(l not in (-100, 0) for l in ex["labels"]):
        print("Sentence №", i)
        print(ex["tokens"])
        print(ex["tags"])
        print(ex["labels"])
        break

NameError: name 'my_train' is not defined